# Ovarian Cancer Detection: Preprocessing & Leakage Check
**Aaishah Khan** · ASDS 5302 · builds on `oc50k_EDA.ipynb`

Goal: handle missing values, scale features, and figure out why the baseline logistic regression scores 100%.

Data: *Ovarian Cancer Risk & Detection* (Kaggle). Download `archive.zip` and put `oc_risk50k_single_dataset.csv` in the same folder as this notebook.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
# If you only have the Kaggle zip (e.g. in Colab), uncomment the next line first
# !unzip -o archive.zip

df = pd.read_csv("oc_risk50k_single_dataset.csv")
print(df.shape)

In [ ]:
# Same cleanup and split as oc50k_EDA.ipynb:
# drop the ID and post-diagnosis columns (data leakage), use the dataset's own Train/Validation partition
cols_removed = ["Patient_ID", "Risk_Category", "Follow_Up_Status",
                "Treatment_Status", "Time_To_Progression"]
df = df.drop(columns=cols_removed)

target = "Ovarian_Cancer_Status"
X = df.drop(columns=[target, "Partition"])
y = df[target]

x_train, y_train = X[df.Partition == "Train"], y[df.Partition == "Train"]
x_test,  y_test  = X[df.Partition == "Validation"], y[df.Partition == "Validation"]
print(x_train.shape, x_test.shape)

## 1. Missing values

In [ ]:
# Count on the training set only, so the validation set stays unseen
missing = x_train.isnull().sum()
missing_pct = (missing[missing > 0] / len(x_train) * 100).sort_values()

ax = missing_pct.plot(kind="barh", color="#7A2E1F", figsize=(8, 3))
ax.set_xlabel("% of training rows missing")
ax.set_title("Missing values by feature (train set)")
plt.tight_layout()
plt.savefig("MissingValues.png", dpi=200)
plt.show()

Only 4 features have missing values, each around 4%. Dropping incomplete rows would lose roughly 15% of the data, so we impute instead.

## 2. Median imputation + scaling

In [ ]:
# Median is robust to the outliers we kept; scaling also fixes the lbfgs convergence warning
def evaluate(X_tr, X_te):
    model = make_pipeline(SimpleImputer(strategy="median"),
                          StandardScaler(),
                          LogisticRegression(max_iter=1000))
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    return {"Accuracy": round(accuracy_score(y_test, pred), 3),
            "Precision": round(precision_score(y_test, pred), 3),
            "Recall": round(recall_score(y_test, pred), 3),
            "F1": round(f1_score(y_test, pred), 3)}

print("All features:   ", evaluate(x_train, x_test))

## 3. Leakage check: drop imaging/tumor features

In [ ]:
# These describe a mass that already exists, so they are the most likely source of leakage
imaging_cols = ["Tumor_Size", "Laterality", "Tumor_Morphology", "Solid_Component",
                "Ascites_Status", "Bilateral_Mass", "Papillary_Projection",
                "Doppler_Vascularity", "Lymph_Node_Enlargement", "Imaging_Risk_Score"]

print("Without imaging:", evaluate(x_train.drop(columns=imaging_cols),
                                   x_test.drop(columns=imaging_cols)))

## 4. Why is it still perfect? Single-feature check

In [ ]:
# ROC-AUC of each feature on its own (0.5 = random, 1.0 = perfect separation)
single_auc = {}
for col in x_test.columns:
    auc = roc_auc_score(y_test, x_test[col].fillna(x_train[col].median()))
    single_auc[col] = max(auc, 1 - auc)

pd.Series(single_auc).sort_values(ascending=False).head(10).round(3)

## Takeaways
- Missing data is small (~4% in 4 features) and handled with median imputation.
- The model still scores 100% after removing the imaging/tumor features, so imaging leakage is **not** the cause.
- Single biomarkers (e.g. CA-125, LDH, HE4) separate the classes almost perfectly on their own: the synthetic data is too clean, so near-perfect scores are expected for any model.
- Next: discuss with Dr. Ozkan whether to keep this dataset (and focus on feature importance / interpretation) or switch to a harder one.